# Decision Intelligence for Nestle: optimizing Marco's diet

Meet **Marco**. Marco runs a startup and runs marathons, so he optimizes everything: burn rate, runway, his 5K splits. His diet, though, he runs on vibes. Being an exec, he built a spreadsheet, and forty tabs later he had a beautiful dashboard that told him, in seven colors, exactly how under-fed he was.

That is Business Intelligence: it told Marco his diet was a disaster. **Decision Intelligence just hands him dinner.**

Marco is a vegan endurance athlete (~70 kg, ~3,000 kcal/day). This notebook runs three questions against a RelationalAI model over real nutrition data (USDA FoodData Central + Open Food Facts) and a synthesized recipe catalog.

In [ ]:
# Snowsight: install RelationalAI (PyRel) and Plotly into the notebook's
# container runtime. Requires the PyPI external access integration on the
# notebook (set in CREATE NOTEBOOK), or add them via the Packages panel.
import sys
!pip install relationalai==1.9.0 plotly

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())  # staged .py files sit beside the notebook
from concurrent.futures import ThreadPoolExecutor
import plotly.graph_objects as go
import demo_queries as dq
from demo_queries import model, Recipe, _build_menu_problem, _menu_df
from relationalai.semantics.std import aggregates as aggs

# relationalai 1.9.0 blocks its sync API on a running event loop (Jupyter),
# so run every query/solve in a worker thread that has no loop.
_ex = ThreadPoolExecutor(max_workers=1)
def run(fn, *a, **k):
    return _ex.submit(fn, *a, **k).result()
print('Loaded. Engines warm on first query.')

Loaded. Engines warm on first query.


## Q1 (Rules) - the problem: cheap looks fine until you check nutrition

If Marco just grabs the cheapest vegan recipe per meal slot, the day is cheap and quietly malnourished.

In [2]:
day, totals, cost = run(dq.q1_rules)
tgt = run(dq.marco_targets).set_index('code')
nutr = ['kcal','protein_g','carb_g','fiber_g','calcium_mg','iron_mg','b12_ug','vitd_ug']
labels, pct, colors = [], [], []
for c in nutr:
    if c in tgt.index and tgt.loc[c,'min_amount'] > 0:
        p = 100.0 * totals[c] / tgt.loc[c,'min_amount']
        labels.append(c); pct.append(round(p,0))
        colors.append('#2ca02c' if p >= 100 else '#d62728')
fig = go.Figure(go.Bar(x=labels, y=pct, marker_color=colors,
                       text=[f'{p:.0f}%' for p in pct], textposition='outside'))
fig.add_hline(y=100, line_dash='dash', annotation_text='target floor')
fig.update_layout(title=f"Q1: Marco's ${cost:.2f} naive day - % of each target met (red = miss)",
                  yaxis_title='% of floor', width=820, height=420)
fig.show()

Q1  RULES / PROBLEM EXPOSURE



Vegan-eligible recipes by meal type:
meal_type recipes
breakfast      73
     main      68
    snack      27

Vegan recipes: 168   Gluten-free recipes: 122



Naive cheapest-per-slot day (1 breakfast + 2 mains + 2 snacks):
meal_type                              name  cost_usd   kcal  protein_g  b12_ug  vitd_ug
breakfast               Banana chia pudding    1.2837 343.10      11.15     2.0      3.0
     main Black beans curry with brown rice    1.3156 727.98      28.36     0.0      0.0
     main Black beans curry with white rice    1.3156 761.28      30.05     0.0      0.0
    snack  Cocoa peanut butter energy balls    0.5280 357.78      10.29     0.0      0.0
    snack        Cocoa almonds energy balls    0.5696 353.98      10.07     0.0      0.0

  Day cost: $5.01
  Nutrition vs Marco's targets:
    kcal           2544.1  (floor  2800.0)  MISS
    protein_g        89.9  (floor    98.0)  MISS
    carb_g          418.2  (floor   420.0)  MISS
    fiber_g          82.7  (floor    35.0)  OK 
    calcium_mg     1029.5  (floor  1000.0)  OK 
    iron_mg          24.4  (floor    14.0)  OK 
    b12_ug            2.0  (floor     2.4)  MISS
    vitd_u

Cheap (about $5/day) but it fails on calories, protein, carbs, B12, and vitamin D. You can't out-train a bad spreadsheet.

## Q2 + Q3 - the fix, then a persistent rule

**Q2 (Prescriptive):** a mixed-integer program picks recipes to fill the slots (1 breakfast, 2 mains, 1-2 snacks), all vegan, meeting all 16 nutrient targets, at minimum cost. **Q3 (Persistent rule):** the operator adds a 3.0 kg/day carbon cap and the same model re-solves instantly. We solve both on one Problem (constraints accumulate) so the decision variable stays consistent.

In [3]:
def solve_q2_q3():
    p = _build_menu_problem('cost_usd')
    p.solve('highs')
    base = _menu_df()
    p.satisfy(model.require(aggs.sum(Recipe.co2e_kg * Recipe.selected) <= 3.0), name=['co2_cap'])
    p.solve('highs')
    capped = _menu_df()
    return base, capped
base, capped = run(solve_q2_q3)
base_cost, base_co2 = base.cost_usd.sum(), base.co2e_kg.sum()
cap_cost, cap_co2 = capped.cost_usd.sum(), capped.co2e_kg.sum()
print(f'Q2 optimal: ${base_cost:.2f}/day, {base.kcal.sum():.0f} kcal, '
      f'{base.protein_g.sum():.0f} g protein, vitD {base.vitd_ug.sum():.1f} ug, CO2 {base_co2:.2f} kg')
print(f'Q3 under 3.0 kg CO2 cap: ${cap_cost:.2f}/day at {cap_co2:.2f} kg '
      f'(+${cap_cost-base_cost:.2f} for {100*(1-cap_co2/base_co2):.0f}% less carbon)')

[Rules created in a loop] Many rules are being created from the same call site, which may indicate a Python loop. Each creates a separate rule that must be processed, which can slow down queries. Consider expressing this as a single rule instead, e.g. using data().


Q2 optimal: $7.06/day, 2806 kcal, 107 g protein, vitD 16.2 ug, CO2 4.38 kg
Q3 under 3.0 kg CO2 cap: $7.17/day at 2.92 kg (+$0.11 for 33% less carbon)


In [4]:
m = base.sort_values('meal_type')
fig = go.Figure(go.Table(
    header=dict(values=['meal','recipe','$','kcal','protein g','B12 ug','vitD ug'],
                fill_color='#13294b', font=dict(color='white')),
    cells=dict(values=[m.meal_type, m.name, m.cost_usd.round(2), m.kcal.round(0),
                       m.protein_g.round(0), m.b12_ug.round(1), m.vitd_ug.round(1)])))
fig.update_layout(title=f'Q2: cheapest compliant day for Marco - ${base_cost:.2f}', width=900, height=320)
fig.show()

In [5]:
fig = go.Figure()
fig.add_bar(name='cost $/day', x=['baseline','CO2-capped'], y=[round(base_cost,2), round(cap_cost,2)], marker_color='#1f77b4')
fig.add_bar(name='CO2 kg/day', x=['baseline','CO2-capped'], y=[round(base_co2,2), round(cap_co2,2)], marker_color='#2ca02c')
fig.update_layout(barmode='group',
                  title=f'Q3: +${cap_cost-base_cost:.2f}/day buys {100*(1-cap_co2/base_co2):.0f}% less carbon (tighter caps go infeasible)',
                  width=720, height=420)
fig.show()

## The takeaway

Same data, same model, three questions: expose the problem, solve it, and let an operator add a rule the model instantly respects. BI showed Marco the problem in seven colors; Decision Intelligence handed him dinner, then a greener dinner when the rule changed. You can't out-train a bad spreadsheet.